In [1]:
from modules.langchain_init import get_llm, get_embeddings
from langchain_community.document_loaders import WebBaseLoader, RecursiveUrlLoader
import bs4


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
llm = get_llm("openai/gpt-4o-mini-2024-07-18")

In [3]:
embeddings = get_embeddings()

In [4]:
import re

from bs4 import BeautifulSoup, SoupStrainer

strainer = SoupStrainer(class_ = 'content')

def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "lxml", parse_only=strainer)
    return re.sub(r"\n\n+", "\n\n", soup.text).strip()


In [11]:
loader = RecursiveUrlLoader(
    "https://book.cairo-lang.org/",
    extractor=bs4_extractor,
    max_depth=2,
)

In [12]:
docs = loader.load()


In [13]:

for doc in docs:
    print(doc.metadata)

{'source': 'https://book.cairo-lang.org/', 'content_type': 'text/html; charset=utf-8', 'title': 'The Cairo Book - The Cairo Programming Language', 'description': '', 'language': 'en'}
{'source': 'https://book.cairo-lang.org/ch14-03-contract-events.html', 'content_type': 'text/html; charset=utf-8', 'title': 'Contract Events - The Cairo Programming Language', 'description': '', 'language': 'en'}
{'source': 'https://book.cairo-lang.org/ch16-05-02-randomness.html', 'content_type': 'text/html; charset=utf-8', 'title': 'Randomness - The Cairo Programming Language', 'description': '', 'language': 'en'}
{'source': 'https://book.cairo-lang.org/ch07-05-separating-modules-into-different-files.html', 'content_type': 'text/html; charset=utf-8', 'title': 'Separating Modules into Different Files - The Cairo Programming Language', 'description': '', 'language': 'en'}
{'source': 'https://book.cairo-lang.org/ch08-00-generic-types-and-traits.html', 'content_type': 'text/html; charset=utf-8', 'title': 'Ge

In [14]:
from langchain_core.vectorstores import InMemoryVectorStore



In [15]:

my_vector_store = InMemoryVectorStore.from_documents(docs, embeddings)

In [16]:
retriever = my_vector_store.as_retriever()

In [17]:
retriever.invoke("What is Cairo?")

[Document(id='3efa8e8c-a7b5-45ee-8187-5eadd4e94335', metadata={'source': 'https://book.cairo-lang.org/ch04-00-understanding-ownership.html', 'content_type': 'text/html; charset=utf-8', 'title': 'Understanding Ownership - The Cairo Programming Language', 'description': '', 'language': 'en'}, page_content="Understanding Cairo's Ownership system\nCairo is a language built around a linear type system that allows us to\nstatically ensure that in every Cairo program, a value is used exactly once.\nThis linear type system helps prevent runtime errors by ensuring that operations that could cause such errors, such as writing twice to a memory cell, are detected at compile time.\nThis is achieved by implementing an ownership system\nand forbidding copying and dropping values by default. In this chapter, we’ll\ntalk about Cairo's ownership system as well as references and snapshots."),
 Document(id='d8934fd7-9246-4503-9713-22e5b2cd8624', metadata={'source': 'https://book.cairo-lang.org/ch02-02-da